# 05 - DQ Audit

In [ ]:
dbutils.widgets.text("curated_base","abfss://olistdata@olistecommdatastorage.dfs.core.windows.net/curated")
CUR=dbutils.widgets.get("curated_base").rstrip("/")

In [ ]:
from pyspark.sql.functions import col,current_timestamp
sv_orders=spark.read.format("delta").load(f"{CUR}/silver/orders")
rows=sv_orders.count()
null_id=sv_orders.where(col("order_id").isNull()).count()
dq=spark.createDataFrame([("silver.orders",rows,null_id)],["table_name","row_count","null_order_id"]).withColumn("run_ts",current_timestamp())
dq.write.format("delta").mode("append").save(f"{CUR}/audit/dq_runs")
spark.read.format("delta").load(f"{CUR}/audit/dq_runs").orderBy("run_ts",ascending=False).show(10,False)